# Notebook 4.2: Full Pipeline — gmsh Geometry → FEniCS Simulation

## Objective
Run the **complete design-to-simulation workflow**:
1. Define geometry parametrically (simulates KLayout export)
2. Generate mesh with gmsh Python API
3. Load mesh into FEniCS
4. Solve Stokes + tracer transport
5. Analyse mixing efficiency

**Geometry:** T-mixer (two inlets, one outlet).

In [ ]:
import gmsh
from fenics import *
import numpy as np
import matplotlib.pyplot as plt
import meshio
import os

%matplotlib inline
set_log_level(LogLevel.WARNING)

---
## Step 1: Create T-Mixer Geometry with gmsh

In [ ]:
# --- Geometry parameters (as if exported from KLayout) ---
w  = 0.10   # Channel width
L  = 1.00   # Main channel length
Lb = 0.30   # Branch length
lc = 0.015  # Mesh characteristic length

msh_file = "/tmp/t_mixer.msh"
xml_file = "/tmp/t_mixer.xml"

gmsh.initialize()
gmsh.model.add("t_mixer")
geo = gmsh.model.geo

# Main channel corners
# Bottom wall: y = -w/2 from x=0 to x=L
# Top wall: y = w/2 — but interrupted at x=L/2 to x=L/2+w for the branch
xb = L / 2  # branch x-position

geo.addPoint(0,      -w/2, 0, lc, 1)   # main bottom-left
geo.addPoint(L,      -w/2, 0, lc, 2)   # main bottom-right
geo.addPoint(L,       w/2, 0, lc, 3)   # main top-right
geo.addPoint(xb+w,    w/2, 0, lc, 4)   # junction top-right
geo.addPoint(xb+w,    w/2+Lb, 0, lc, 5) # branch top-right
geo.addPoint(xb,      w/2+Lb, 0, lc, 6) # branch top-left
geo.addPoint(xb,      w/2, 0, lc, 7)   # junction top-left
geo.addPoint(0,       w/2, 0, lc, 8)   # main top-left

for i in range(1, 8):
    geo.addLine(i, i+1, i)
geo.addLine(8, 1, 8)

geo.addCurveLoop(list(range(1, 9)), 1)
geo.addPlaneSurface([1], 1)

# Physical groups
gmsh.model.addPhysicalGroup(1, [8], name="inlet_main")
gmsh.model.addPhysicalGroup(1, [5], name="inlet_branch")
gmsh.model.addPhysicalGroup(1, [2], name="outlet")
gmsh.model.addPhysicalGroup(1, [1, 3, 4, 6, 7], name="walls")
gmsh.model.addPhysicalGroup(2, [1], name="fluid")

geo.synchronize()
gmsh.model.mesh.generate(2)
gmsh.model.mesh.setOrder(2)
gmsh.write(msh_file)
gmsh.finalize()

print(f"Mesh written to {msh_file}")

---
## Step 2: Convert and Load Mesh into FEniCS

In [ ]:
# Convert .msh → .xml for FEniCS legacy
msh = meshio.read(msh_file)
# Keep only triangle cells
cells = {"triangle": msh.cells_dict["triangle"]}
mesh_io = meshio.Mesh(points=msh.points[:, :2], cells=cells)
meshio.write(xml_file, mesh_io)

mesh = Mesh(xml_file)
print(f"Mesh loaded: {mesh.num_cells()} cells, {mesh.num_vertices()} vertices")

---
## Step 3: Solve Stokes in the T-Mixer

In [ ]:
mu = 1.0
U_in = 0.1   # Inlet velocity
tol = 1e-8

W = FunctionSpace(mesh, MixedElement([
    VectorElement("P", mesh.ufl_cell(), 2),
    FiniteElement("P", mesh.ufl_cell(), 1)
]))

# Inlets: uniform flow inward; walls: no-slip; outlet: free (natural BC)
bcs = [
    DirichletBC(W.sub(0), Constant((U_in, 0.0)),  # main inlet → right
                f"on_boundary && x[0] < {tol}"),
    DirichletBC(W.sub(0), Constant((0.0, -U_in)), # branch inlet → down
                f"on_boundary && x[1] > {w/2 + Lb - tol}"),
    DirichletBC(W.sub(0), Constant((0.0, 0.0)),    # walls
                f"on_boundary && x[0] > {tol} && x[0] < {L-tol} && x[1] < {w/2+Lb-tol}"),
]

(u, p) = TrialFunctions(W)
(v, q) = TestFunctions(W)

a = (mu * inner(grad(u), grad(v)) - p*div(v) + q*div(u)) * dx
L_f = dot(Constant((0.0, 0.0)), v) * dx

w_sol = Function(W)
solve(a == L_f, w_sol, bcs)
u_flow, p_flow = w_sol.split()

print(f"Stokes solved. Max |u| = {u_flow.vector().norm('linf'):.5f}")

---
## Step 4: Tracer Transport — Two Inlets, One Species Each

In [ ]:
D  = 0.001
dt = 0.1
T  = 5.0

S   = FunctionSpace(mesh, "P", 1)
c   = TrialFunction(S)
phi = TestFunction(S)
c_n = Function(S)

# Main inlet c=1, branch inlet c=0  → mixing at junction
bc_main   = DirichletBC(S, Constant(1.0), f"on_boundary && x[0] < {tol}")
bc_branch = DirichletBC(S, Constant(0.0), f"on_boundary && x[1] > {w/2+Lb-tol}")
bc_c = [bc_main, bc_branch]

a_c = (c/dt * phi + dot(u_flow, grad(c))*phi + D * dot(grad(c), grad(phi))) * dx
L_c = c_n/dt * phi * dx

A_c = assemble(a_c)
for bc in bc_c: bc.apply(A_c)

c_sol = Function(S)
t = 0
while t < T - 1e-8:
    t += dt
    b_c = assemble(L_c)
    for bc in bc_c: bc.apply(b_c)
    solve(A_c, c_sol.vector(), b_c)
    c_n.assign(c_sol)

print(f"Transport solved. Final t = {t:.1f}")

---
## Step 5: Visualise

In [ ]:
S_scalar = FunctionSpace(mesh, "P", 1)
u_mag = project(sqrt(u_flow[0]**2 + u_flow[1]**2), S_scalar)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
c1 = plot(u_mag, ax=ax, cmap='viridis')
plt.colorbar(c1, ax=ax, label='$|u|$')
ax.set_title('Velocity magnitude', fontsize=12)

ax = axes[1]
c2 = plot(c_sol, ax=ax, cmap='RdYlBu_r', vmin=0, vmax=1)
plt.colorbar(c2, ax=ax, label='$c$')
ax.set_title(f'Tracer concentration at t={T:.0f}', fontsize=12)

plt.tight_layout()
plt.savefig('/tmp/t_mixer_result.png', dpi=150)
plt.show()

---
## Summary

Full pipeline achieved:
- **gmsh** (parametric geometry) → mesh
- **meshio** (format conversion)
- **FEniCS** (Stokes + transport)
- **matplotlib** (visualisation)

**For a real KLayout workflow:** export GDS, parse polygons with `gdspy`, feed coordinates to the `gmsh.model.geo.addPoint()` calls above.

**Exercise:** Try varying the branch width `w_branch` vs. the main channel width. How does that affect mixing length?